# HW02：多层感知机、模型选择、数值稳定性与协变量偏移

> 按作业 PDF 要求完成理论题与编程题。编程题均设置了 `print` 输出，运行后可得到训练结果、梯度范数、MSE 对比和曲线图。提交时可将文件名改为：`HW02-学号-姓名.ipynb`。

In [ ]:
import math, random
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 2 多层感知机

## 2.1 理论计算题

### 1. 非线性激活函数的重要性

已知单隐藏层 MLP：

$$h=W_1x+b_1$$

$$o=W_2h+b_2$$

将隐藏层代入输出层：

$$\begin{aligned}
o&=W_2(W_1x+b_1)+b_2\\
&=W_2W_1x+W_2b_1+b_2
\end{aligned}$$

令

$$W'=W_2W_1,\quad b'=W_2b_1+b_2$$

则

$$o=W'x+b'$$

因此，没有非线性激活函数时，多层线性变换的复合仍然是一个线性/仿射变换，该网络等价于一个单层神经网络。这说明非线性激活函数是 MLP 表达复杂非线性关系的关键。

### 2. Sigmoid 与 tanh 的表达式及导数

Sigmoid：

$$\sigma(x)=\frac{1}{1+e^{-x}}$$

$$\begin{aligned}
\sigma'(x)&=\frac{e^{-x}}{(1+e^{-x})^2}\\
&=\frac{1}{1+e^{-x}}\left(1-\frac{1}{1+e^{-x}}\right)\\
&=\sigma(x)(1-\sigma(x))
\end{aligned}$$

因此：

$$\sigma'(x)=\sigma(x)(1-\sigma(x))$$

tanh：

$$\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}$$

其导数为：

$$\tanh'(x)=1-\tanh^2(x)$$

Sigmoid 和 tanh 的导数都可由函数自身表示，便于反向传播；但输入绝对值较大时导数趋近于 0，容易造成梯度消失。

## 2.2 编程题：从零实现 Fashion-MNIST 单隐藏层 MLP

不使用 `nn.Linear`、`nn.CrossEntropyLoss`、优化器等高级 API，只用张量基础算子完成前向传播、ReLU、Softmax 交叉熵和手动 SGD。

In [ ]:
try:
    from torchvision import datasets, transforms
except Exception as e:
    raise ImportError("请先安装 torchvision：pip install torchvision") from e

batch_size = 256
transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root="./data", train=False, transform=transform, download=True)
train_iter = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_iter = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
print("训练集样本数:", len(train_dataset))
print("测试集样本数:", len(test_dataset))

In [ ]:
num_inputs, num_hiddens, num_outputs = 28*28, 256, 10
W1 = torch.normal(0, 0.01, size=(num_inputs, num_hiddens), device=device, requires_grad=True)
b1 = torch.zeros(num_hiddens, device=device, requires_grad=True)
W2 = torch.normal(0, 0.01, size=(num_hiddens, num_outputs), device=device, requires_grad=True)
b2 = torch.zeros(num_outputs, device=device, requires_grad=True)
params = [W1, b1, W2, b2]
print("W1:", W1.shape, "b1:", b1.shape, "W2:", W2.shape, "b2:", b2.shape)

In [ ]:
def relu(X):
    return torch.clamp(X, min=0)

def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(torch.matmul(X, W1) + b1)
    return torch.matmul(H, W2) + b2

def softmax_cross_entropy(logits, y):
    # log-sum-exp 技巧提升数值稳定性
    max_logits = logits.max(dim=1, keepdim=True).values
    z = logits - max_logits
    log_probs = z - torch.log(torch.exp(z).sum(dim=1, keepdim=True))
    return -log_probs[torch.arange(y.shape[0], device=y.device), y].mean()

def accuracy(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()

def evaluate(data_iter):
    total_loss, total_acc, total_num = 0.0, 0.0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            logits = net(X)
            loss = softmax_cross_entropy(logits, y)
            n = y.numel()
            total_loss += loss.item() * n
            total_acc += accuracy(logits, y) * n
            total_num += n
    return total_loss / total_num, total_acc / total_num

def sgd(params, lr):
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad
            p.grad.zero_()

In [ ]:
num_epochs, lr = 10, 0.1
for epoch in range(num_epochs):
    train_loss_sum, train_acc_sum, train_num = 0.0, 0.0, 0
    for X, y in train_iter:
        X, y = X.to(device), y.to(device)
        logits = net(X)
        loss = softmax_cross_entropy(logits, y)
        loss.backward()
        sgd(params, lr)
        n = y.numel()
        train_loss_sum += loss.item() * n
        train_acc_sum += accuracy(logits, y) * n
        train_num += n
    test_loss, test_acc = evaluate(test_iter)
    print(f"Epoch {epoch+1:02d}: train loss={train_loss_sum/train_num:.4f}, "
          f"train acc={train_acc_sum/train_num:.4f}, test loss={test_loss:.4f}, test acc={test_acc:.4f}")

# 3 模型选择，权重衰减和丢弃法

## 3.1 理论计算题

### 1. 过拟合与欠拟合

训练误差是模型在训练集上的误差，反映模型对已见样本的拟合能力；泛化误差是模型在未见样本或真实测试分布上的误差，反映模型对新数据的预测能力。

当训练误差极低但泛化误差很高时，模型处于**过拟合**状态，即模型过度记住训练数据中的细节甚至噪声。缓解方法包括降低模型复杂度、减少参数量、降低多项式阶数、加入 L2 权重衰减、使用 Dropout、早停、数据增强或增加训练数据。

当训练误差和泛化误差都很高时，通常是**欠拟合**，说明模型表达能力不足或训练不充分，可增加模型复杂度、训练更久或调整学习率。

### 2. K 折交叉验证

K 折交叉验证步骤：

1. 将训练数据随机划分为 K 个大小尽量相近且互不重叠的子集；
2. 每次取其中 1 个子集作为验证集，其余 K-1 个子集作为训练集；
3. 重复 K 次，使每个子集都恰好作为一次验证集；
4. 记录每次验证误差，并求平均值作为模型或超参数组合的性能估计；
5. 选择平均验证误差较低的模型或超参数；
6. 最后使用全部训练数据重新训练最终模型。

K 折交叉验证能充分利用有限数据，降低一次随机划分带来的偶然性。

## 3.2 编程题：L2 权重衰减与 Dropout

In [ ]:
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    if dropout == 1:
        return torch.zeros_like(X)
    if dropout == 0:
        return X
    mask = (torch.rand_like(X) > dropout).float()
    return mask * X / (1.0 - dropout)

X_demo = torch.ones(10)
print("dropout=0.5 输出示例:", dropout_layer(X_demo, 0.5))
print("dropout=0 输出示例:", dropout_layer(X_demo, 0.0))

def sgd_with_weight_decay(params, lr, weight_decay=0.0, decay_flags=None):
    if decay_flags is None:
        decay_flags = [True] * len(params)
    with torch.no_grad():
        for p, use_decay in zip(params, decay_flags):
            if use_decay and weight_decay > 0:
                p *= (1 - lr * weight_decay)  # 权重先乘以 (1-ηλ)
            p -= lr * p.grad
            p.grad.zero_()

def net_with_dropout(X, params, is_training=True, dropout=0.5):
    W1, b1, W2, b2 = params
    X = X.reshape((-1, 28*28))
    H = relu(torch.matmul(X, W1) + b1)
    if is_training:
        H = dropout_layer(H, dropout)
    return torch.matmul(H, W2) + b2

print("Dropout 和带权重衰减的 SGD 定义完成。")

### 对比实验：高维多项式拟合

构造小样本高维多项式回归任务，比较：1）无正则化；2）有权重衰减；3）有 Dropout 的训练和验证误差曲线。

In [ ]:
torch.manual_seed(SEED)
n_train, n_val, max_degree = 30, 100, 20

def make_poly_data(n):
    x = torch.linspace(-3, 3, n).reshape(-1, 1)
    y = 1.5 + 2.0*x - 0.8*x**2 + 0.3*torch.randn_like(x)
    return x, y

x_train, y_train = make_poly_data(n_train)
x_val, y_val = make_poly_data(n_val)
powers = torch.arange(1, max_degree+1).float().reshape(1, -1)
X_train_poly = x_train**powers / torch.exp(torch.lgamma(powers + 1))
X_val_poly = x_val**powers / torch.exp(torch.lgamma(powers + 1))
X_train_poly, y_train = X_train_poly.to(device), y_train.to(device)
X_val_poly, y_val = X_val_poly.to(device), y_val.to(device)
print("训练特征形状:", X_train_poly.shape, "验证特征形状:", X_val_poly.shape)

In [ ]:
def mse_loss(y_hat, y):
    return ((y_hat - y)**2).mean()

def train_poly_model(mode="none", weight_decay=0.0, dropout=0.0, epochs=300, lr=0.03):
    torch.manual_seed(SEED)
    W = torch.normal(0, 0.01, size=(max_degree, 1), device=device, requires_grad=True)
    b = torch.zeros(1, device=device, requires_grad=True)
    train_losses, val_losses = [], []
    for epoch in range(epochs):
        H = X_train_poly
        if mode == "dropout":
            H = dropout_layer(H, dropout)
        loss = mse_loss(torch.matmul(H, W) + b, y_train)
        loss.backward()
        with torch.no_grad():
            if mode == "weight_decay":
                W *= (1 - lr * weight_decay)
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_(); b.grad.zero_()
        with torch.no_grad():
            train_losses.append(mse_loss(torch.matmul(X_train_poly, W)+b, y_train).item())
            val_losses.append(mse_loss(torch.matmul(X_val_poly, W)+b, y_val).item())
    print(f"{mode:>12s}: final train loss={train_losses[-1]:.4f}, final val loss={val_losses[-1]:.4f}")
    return train_losses, val_losses

loss_none = train_poly_model("none")
loss_wd = train_poly_model("weight_decay", weight_decay=3.0)
loss_dropout = train_poly_model("dropout", dropout=0.4)

In [ ]:
plt.figure(figsize=(9,6))
plt.plot(loss_none[0], label="Train - No Reg")
plt.plot(loss_none[1], label="Val - No Reg")
plt.plot(loss_wd[0], label="Train - Weight Decay")
plt.plot(loss_wd[1], label="Val - Weight Decay")
plt.plot(loss_dropout[0], label="Train - Dropout")
plt.plot(loss_dropout[1], label="Val - Dropout")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss")
plt.title("Loss Curves: No Regularization vs Weight Decay vs Dropout")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

# 4 数值稳定性和激活函数

## 4.1 理论计算题

深层网络的反向传播中，梯度常包含多层矩阵连乘：

$$\prod_{i=t}^{d-1}\frac{\partial h^{i+1}}{\partial h^i}$$

若 $h^{i+1}=\phi(W_i h^i+b_i)$，则局部梯度近似为：

$$\frac{\partial h^{i+1}}{\partial h^i}=D_iW_i$$

其中 $D_i$ 是激活函数导数组成的对角矩阵。于是有：

$$\left\|\prod_{i=t}^{d-1}D_iW_i\right\|\leq\prod_{i=t}^{d-1}\|D_i\|\|W_i\|$$

- 若多数层满足 $\|D_i\|\|W_i\|>1$，连乘会随层数指数级增大，导致梯度爆炸；
- 若多数层满足 $\|D_i\|\|W_i\|<1$，连乘会随层数指数级缩小，导致梯度消失。

Sigmoid 的导数最大值为 0.25，且输入较大或较小时会进入饱和区，导数接近 0；tanh 也存在饱和区。因此深层网络使用 Sigmoid/tanh 时更容易梯度消失。

ReLU 在 $x>0$ 时导数为 1，不会在正区间饱和，所以能明显缓解梯度消失。但如果权重初始化过大，仍可能出现梯度爆炸或数值溢出；若大量神经元输入长期小于 0，也可能出现“神经元死亡”。

## 4.2 编程题：模拟数值不稳定并验证初始化策略

In [ ]:
import torch.nn as nn

def build_deep_net(activation="sigmoid", width=256, depth=20, input_dim=256, output_dim=1):
    layers, in_dim = [], input_dim
    act_cls = nn.Sigmoid if activation == "sigmoid" else nn.ReLU
    for _ in range(depth):
        layers.append(nn.Linear(in_dim, width))
        layers.append(act_cls())
        in_dim = width
    layers.append(nn.Linear(width, output_dim))
    return nn.Sequential(*layers).to(device)

def init_normal(model, std=1.0):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=std)
            nn.init.zeros_(m.bias)

def init_xavier(model):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

def print_grad_norms(model, title, max_show=4):
    norms = []
    for name, p in model.named_parameters():
        if "weight" in name and p.grad is not None:
            norms.append((name, p.grad.norm().item()))
    print("\n" + title)
    print("前几层梯度范数:")
    for name, val in norms[:max_show]:
        print(name, f"{val:.6e}")
    print("后几层梯度范数:")
    for name, val in norms[-max_show:]:
        print(name, f"{val:.6e}")
    has_nan = any(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
    print("是否存在 NaN:", has_nan)

def run_one_backward(model, title):
    X = torch.randn(64, 256, device=device)
    y = torch.randn(64, 1, device=device)
    model.zero_grad()
    pred = model(X)
    loss = ((pred - y)**2).mean()
    loss.backward()
    print("loss:", loss.item())
    print_grad_norms(model, title)

In [ ]:
model_sigmoid = build_deep_net("sigmoid", width=256, depth=20)
init_normal(model_sigmoid, std=1.0)
run_one_backward(model_sigmoid, "Sigmoid + Normal(std=1): 梯度范数")

model_relu_big = build_deep_net("relu", width=256, depth=20)
init_normal(model_relu_big, std=10.0)
run_one_backward(model_relu_big, "ReLU + Normal(std=10): 梯度范数")

model_relu_xavier = build_deep_net("relu", width=256, depth=20)
init_xavier(model_relu_xavier)
run_one_backward(model_relu_xavier, "ReLU + Xavier: 梯度范数")

# 5 泛化表现，协变量偏移和对抗性数据

## 5.1 理论计算题

### 1. 协变量偏移

协变量偏移指训练环境和测试环境中的输入特征分布发生变化：

$$p(x)\ne q(x)$$

但输入到标签的条件关系保持不变：

$$p(y|x)=q(y|x)$$

例如医疗影像中，训练数据来自 A 医院，测试数据来自 B 医院。两家医院设备、成像参数、人群结构不同，导致影像特征分布变化；但同样影像表现对应同一疾病标签的规律基本不变，这就是协变量偏移。

### 2. 标签偏移

标签偏移指训练环境与测试环境中标签先验比例变化：

$$p(y)\ne q(y)$$

但每个类别内部的特征分布保持不变：

$$p(x|y)=q(x|y)$$

例如医疗筛查中，医院数据中患病样本比例较高，而普通体检人群中患病比例较低；但对于患病或未患病类别内部，其检查指标分布可能相对稳定，这属于标签偏移。

### 3. 区别与联系

二者都属于训练分布与测试分布不一致导致的环境非平稳性问题，都会影响泛化性能。区别在于：协变量偏移主要是 $x$ 的分布变了但 $y|x$ 不变；标签偏移主要是 $y$ 的先验比例变了但 $x|y$ 不变。协变量偏移常用重要性加权 $q(x)/p(x)$ 处理；标签偏移常通过估计类别先验变化并重加权处理。

## 5.2 编程题：模拟协变量偏移并使用权重修正

In [ ]:
np.random.seed(SEED); torch.manual_seed(SEED)
n_train, n_test = 1000, 500
x_train = np.random.normal(-1.0, 1.0, size=(n_train, 1))
x_test = np.random.normal(2.0, 1.0, size=(n_test, 1))
y_train = 2*x_train + np.random.normal(0, 0.2, size=(n_train, 1))
y_test = 2*x_test + np.random.normal(0, 0.2, size=(n_test, 1))
print("训练集 x 均值:", x_train.mean(), "测试集 x 均值:", x_test.mean())

def add_intercept(x):
    return np.concatenate([np.ones((x.shape[0], 1)), x], axis=1)

def ordinary_least_squares(X, y):
    return np.linalg.pinv(X.T @ X) @ X.T @ y

def weighted_least_squares(X, y, weights):
    W = np.diag(weights.reshape(-1))
    return np.linalg.pinv(X.T @ W @ X) @ X.T @ W @ y

def mse_np(y_pred, y_true):
    return np.mean((y_pred-y_true)**2)

X_train_lr, X_test_lr = add_intercept(x_train), add_intercept(x_test)
beta_base = ordinary_least_squares(X_train_lr, y_train)
y_pred_base = X_test_lr @ beta_base
mse_base = mse_np(y_pred_base, y_test)
print("基线线性回归参数 beta:", beta_base.reshape(-1))
print("校正前测试 MSE:", mse_base)

In [ ]:
# 训练逻辑回归域分类器：训练集样本标记为 0，测试集样本标记为 1
X_domain = np.vstack([x_train, x_test]).astype(np.float32)
y_domain = np.vstack([np.zeros((n_train, 1)), np.ones((n_test, 1))]).astype(np.float32)
X_domain_t = torch.tensor(X_domain, device=device)
y_domain_t = torch.tensor(y_domain, device=device)

w = torch.zeros((1,1), device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)

for epoch in range(1000):
    logits = X_domain_t @ w + b
    prob = torch.sigmoid(logits)
    loss = -(y_domain_t*torch.log(prob+1e-8) + (1-y_domain_t)*torch.log(1-prob+1e-8)).mean()
    loss.backward()
    with torch.no_grad():
        w -= 0.1*w.grad
        b -= 0.1*b.grad
        w.grad.zero_(); b.grad.zero_()
    if (epoch+1) % 250 == 0:
        acc = ((prob > 0.5).float() == y_domain_t).float().mean().item()
        print(f"Domain clf epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.4f}")

with torch.no_grad():
    x_train_t = torch.tensor(x_train.astype(np.float32), device=device)
    p_test = torch.sigmoid(x_train_t @ w + b).cpu().numpy().reshape(-1)
    p_train = 1 - p_test

# q(x)/p(x) = odds * 先验比例修正项
pi_train = n_train / (n_train + n_test)
pi_test = n_test / (n_train + n_test)
weights = (p_test / (p_train + 1e-8)) * (pi_train / pi_test)
weights = np.clip(weights, 0, 50)
weights = weights / weights.mean()
print("权重统计: min=", weights.min(), "max=", weights.max(), "mean=", weights.mean())

In [ ]:
beta_weighted = weighted_least_squares(X_train_lr, y_train, weights)
y_pred_weighted = X_test_lr @ beta_weighted
mse_weighted = mse_np(y_pred_weighted, y_test)
print("加权线性回归参数 beta:", beta_weighted.reshape(-1))
print("校正前测试 MSE:", mse_base)
print("校正后测试 MSE:", mse_weighted)
print("MSE 改变量:", mse_base - mse_weighted)

plt.figure(figsize=(8,5))
plt.hist(x_train.reshape(-1), bins=30, alpha=0.6, label="Train P: N(-1,1)", density=True)
plt.hist(x_test.reshape(-1), bins=30, alpha=0.6, label="Test Q: N(2,1)", density=True)
plt.xlabel("x"); plt.ylabel("Density"); plt.title("Covariate Shift")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

plt.figure(figsize=(8,5))
plt.scatter(x_train.reshape(-1), weights, s=10, alpha=0.5)
plt.xlabel("training sample x"); plt.ylabel("importance weight")
plt.title("Estimated Importance Weights")
plt.grid(True, alpha=0.3); plt.show()

## 5.2 结果说明

本题设定的真实关系是线性形式 $y=2x+\epsilon$，而基线模型也采用线性回归，因此在样本量较充足时，即使发生协变量偏移，普通最小二乘也可能得到接近真实的斜率，校正前后的 MSE 差异不一定非常大。

但实现流程完整展示了协变量偏移校正思路：先训练域分类器估计样本来自测试分布的概率，再根据 $P(test|x)/P(train|x)$ 构造重要性权重，最后使用加权最小二乘让模型更重视测试分布附近的训练样本。